In [2]:
import pandas as pd
import os

### Reading the data in and creating dataframes

In [3]:
# read all data in the data folder; will take ~4 minutes to run 
path = "data/"

xl = pd.ExcelFile(os.path.join(path, "2015-2025 SPD Calls with Close Codes.xlsx")) 

# for the SPD Outcomes data, which includes an unnessary "KEY" sheet that we will ignore
sheets = [sheet for sheet in xl.sheet_names if sheet != "KEY"]

print("Reading CSV")
print('2015...')
eug_cad2015 = pd.read_csv(os.path.join(path, "EugeneCAD2015noloc.csv"))
print('2016...')
eug_cad2016 = pd.read_csv(os.path.join(path, "EugeneCAD2016noloc.csv"))
print('2017...')
eug_cad2017 = pd.read_csv(os.path.join(path, "EugeneCAD2017noloc.csv"))
print('2018...')
eug_cad2018 = pd.read_csv(os.path.join(path, "EugeneCAD2018noloc.csv"))
print('2019...')
eug_cad2019 = pd.read_csv(os.path.join(path, "EugeneCAD2019noloc.csv"))
print('2020...')
eug_cad2020 = pd.read_csv(os.path.join(path, "EugeneCAD2020noloc.csv"))
print('2021...')
eug_cad2021 = pd.read_csv(os.path.join(path, "EugeneCAD2021noloc.csv"), low_memory=False)
print('2022...')
eug_cad2022 = pd.read_csv(os.path.join(path, "EugeneCAD2022noloc.csv"))
print('2023...')
eug_cad2023 = pd.read_csv(os.path.join(path, "EugeneCAD2023noloc.csv"))
print('2024...')
eug_cad2024 = pd.read_csv(os.path.join(path, "EugeneCAD2024noloc.csv"))
print('2025...')
eug_cad2025 = pd.read_csv(os.path.join(path, "EugeneCAD2025noloc.csv"), low_memory=False)

print("Reading Excel")
print('SPD Calls for Service...')
spd_calls = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls for Service.xlsx'), sheet_name=None), ignore_index=True)
print('SPD Responding Units...')
spd_units = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Responding Units.xlsx'), sheet_name=None), ignore_index=True)
print("SPD Outcomes...")
spd_outcomes = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls with Close Codes.xlsx'), sheet_name=sheets), ignore_index=True)
print("Done")

Reading CSV
2015...
2016...
2017...
2018...
2019...
2020...
2021...
2022...
2023...
2024...
2025...
Reading Excel
SPD Calls for Service...
SPD Responding Units...
SPD Outcomes...
Done


In [4]:
dfs = [
    eug_cad2015, eug_cad2016, eug_cad2017, eug_cad2018,
    eug_cad2019, eug_cad2020, eug_cad2021, eug_cad2022,
    eug_cad2023, eug_cad2024, eug_cad2025
]

base_cols = set(dfs[0].columns)

for i, df in enumerate(dfs):
    cols = set(df.columns)
    missing = base_cols - cols
    extra = cols - base_cols
    
    if missing or extra:
        print(f"DataFrame {2015 + i}:")
        if missing:
            print("  Missing:", missing)
        if extra:
            print("  Extra:", extra)

# drop that extra column
eug_cad2025 = eug_cad2025.drop(columns={"month"})

DataFrame 2025:
  Extra: {'month'}


In [5]:
eug_cad = eug_cad2015
for df in dfs[1:]:
    eug_cad = pd.concat([eug_cad, df], ignore_index=True)

eug_cad.columns
eug_cad.head()
print(eug_cad.shape)

(1446014, 20)


### Cleaning Eugene CAD Dataset

#### 1. Subset to the columns I need

In [6]:
columns = ["calltime", "nature", "closed_as", "primeunit"]
eug = eug_cad[columns].copy()

eug

,calltime,nature,closed_as,primeunit
0,2015-01-01 00:00:00.000,PERSON STOP,ASSISTED,_5E48
1,2015-01-01 00:00:44.000,FIGHT,RESOLVED,_3F65
2,2015-01-01 00:01:05.000,CHECK WELFARE,ASSISTED,_3J79
3,2015-01-01 00:03:16.000,SHOTS FIRED,PATROL CHECK,_5E48
4,2015-01-01 00:03:34.000,ILLEGAL FIREWORKS,ADVISED,_5K97
...,...,...,...,...
1446009,2025-12-31 23:39:12.000,TRAFFIC STOP,SOBRIETY CHECK,_4U41
1446010,2025-12-31 23:40:48.000,TRAFFIC STOP,UNIFORM TRAFFIC CITATION ISSUED,_5E56
1446011,2025-12-31 23:45:57.000,PATROL CHECK,UNIFORM TRAFFIC CITATION ISSUED,_4E48
1446012,2025-12-31 23:49:12.000,ASSIST OREGON STATE POLICE,ASSISTED,_5T81


#### 2. Turn ```calltime``` into a datetime object

In [7]:
eug["calltime"] = pd.to_datetime(eug["calltime"])
eug.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1446014 entries, 0 to 1446013
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype         
---  ------     --------------    -----         
 0   calltime   1446014 non-null  datetime64[ns]
 1   nature     1445961 non-null  object        
 2   closed_as  1422086 non-null  object        
 3   primeunit  1091731 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 44.1+ MB


#### 3. Subset dataset further to only have welfare check calls

In [8]:
eug = eug[(eug["nature"] == "CHECK WELFARE") | (eug["nature"] == "CHECK WELFARE, CAHOOTS")]
eug

,calltime,nature,closed_as,primeunit
2,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,_3J79
27,2015-01-01 00:38:37,CHECK WELFARE,ASSISTED,_3J79
47,2015-01-01 01:28:40,CHECK WELFARE,UNABLE TO LOCATE,_5B44
55,2015-01-01 01:41:40,CHECK WELFARE,ASSISTED,_3J79
186,2015-01-01 15:09:21,CHECK WELFARE,ASSISTED,_3J79
...,...,...,...,...
1445916,2025-12-31 17:12:02,CHECK WELFARE,DISREGARDED BY DISPATCH,NaN
1445927,2025-12-31 17:56:18,CHECK WELFARE,UNABLE TO LOCATE,_4U72
1445946,2025-12-31 20:23:13,CHECK WELFARE,WELFARE CHECK DONE,_5E56
1445957,2025-12-31 20:50:01,CHECK WELFARE,DISREGARD,_4U72


### Cleaning Springfield Call Dataset

#### 1. Join SPD Calls and Outcome sets

In [15]:
spd_outcomes

,Incident Number,Close Code
0,15009604,
1,15010887,
2,15019248,
3,15025227,
4,15025271,
...,...,...
567649,25308708,XRPT
567650,25310022,XRPT
567651,25317200,XRPT
567652,25320899,XRPT


In [9]:
spd = spd_calls.merge(spd_outcomes, on="Incident Number")
spd

,Incident Number,Initial Call Type,Final Call Type,Responding Agency,Primary Responding Unit,Call Creation Time,First Dispatched Time,First Arrival Time,Clear Time,Priority,Call Creation Mechanism,Close Code
0,15000074,ALMAUD,AUDIBLE ALARM,SPD,1S18,2015-01-01 01:16:21,2015-01-01 01:18:28,2015-01-01 01:21:13,2015-01-01 01:32:08,3,PHONE,BLDS
1,15000083,TRFSTP,DWS,SPD,3D2,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:34:57,4,SELF,UTC
2,15000167,MVAUNK,MOTOR VEH ACC UNKNOWN INJ,SPD,,2015-01-01 03:15:28,NaT,NaT,NaT,3,E911,DUP
3,15000216,TRFSTP,DWS,SPD,1S22,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:38:53,4,SELF,UTC
4,15000222,SUSPVE,SUSPICIOUS VEHICLE,SPD,1S11,2015-01-01 05:52:53,2015-01-01 05:53:57,2015-01-01 05:56:21,2015-01-01 06:04:04,3,PHONE,INFO
...,...,...,...,...,...,...,...,...,...,...,...,...
567649,25326475,DISVEH,DISABLED VEHICLE,SPD,1S23,2025-12-30 21:15:10,2025-12-30 21:15:10,2025-12-30 21:15:10,2025-12-30 21:17:20,4,SELF,RSLV
567650,25326480,TRFSTP,TRAFFIC STOP,SPD,1S23,2025-12-30 21:25:51,2025-12-30 21:25:51,2025-12-30 21:25:51,2025-12-30 21:30:21,6,SELF,WARN
567651,25326519,CHKWLF,CHECK WELFARE,SPD,3J81,2025-12-30 22:24:34,2025-12-30 22:25:32,2025-12-30 22:31:12,2025-12-30 22:47:29,7,PHONE,ASST
567652,25326528,ALMAUD,AUDIBLE ALARM,SPD,1S21,2025-12-30 22:34:54,2025-12-30 22:41:36,2025-12-30 22:44:28,2025-12-30 22:48:25,3,PHONE,FALS


#### 2. Subset to columns I need and rename

In [10]:
columns = ["Final Call Type", "Call Creation Time", "Primary Responding Unit", "Close Code"]
spd = spd[columns].copy()

spd.rename(columns={
    "Final Call Type": "nature",
    "Call Creation Time": "calltime",
    "Primary Responding Unit": "primeunit",
    "Close Code": "closed_as"
}, inplace=True)

spd = spd[spd["nature"] == "CHECK WELFARE"]

#### 3. Turn ```calltime``` into a datetime object

In [11]:
spd["calltime"] = pd.to_datetime(spd["calltime"])
spd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35076 entries, 365 to 567651
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   nature     35076 non-null  object        
 1   calltime   35076 non-null  datetime64[ns]
 2   primeunit  35076 non-null  object        
 3   closed_as  35076 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 1.3+ MB


#### 4. Clean the ```primeunit``` and ```closed_as``` column

In [12]:
spd["primeunit"] = spd["primeunit"].str.strip()
spd["closed_as"] = spd["closed_as"].str.strip()

In [13]:
spd

,nature,calltime,primeunit,closed_as
365,CHECK WELFARE,2015-01-04 19:27:33,3S21,WELC
431,CHECK WELFARE,2015-01-01 04:07:43,1S22,ASST
439,CHECK WELFARE,2015-01-01 12:32:22,2S11,UTL
456,CHECK WELFARE,2015-01-02 09:12:29,2S11,UNFD
526,CHECK WELFARE,2015-01-02 16:06:16,3S11,REPT
...,...,...,...,...
567620,CHECK WELFARE,2025-12-30 21:36:23,3J81,UTL
567625,CHECK WELFARE,2025-12-29 17:16:57,3J81,ASST
567630,CHECK WELFARE,2025-12-30 09:37:10,2S11,WARN
567641,CHECK WELFARE,2025-12-30 14:10:27,,REL


### CSV Output

In [14]:
eug.to_csv("cleaned_eug.csv", index=False)
spd.to_csv("cleaned_spd.csv", index=False)